# Assignment 30: Chat Groq RAG Application using StreamlitUI
---

# PART 1 — Groq Setup & Basic Chat
## Task 1: ChatGroq Setup

In [ ]:
from dotenv import load_dotenv

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
load_dotenv()

In [ ]:
model = 'meta-llama/llama-prompt-guard-2-22m'

In [ ]:

llm = ChatGroq(
    model=model,
    temperature=0
)

print("ChatGroq initialized successfully.")

## Task 2 — Basic Chat with ChatGroq

In [ ]:
response = llm.invoke("What is machine learning?")

print(response.content)

# PART 2 — RAG Backend Pipeline
## Task 3: Document Loading & Text Splitting

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
loader = PyPDFLoader("document.pdf")

documents = loader.load()

In [ ]:
print("Number of pages:", len(documents))

In [ ]:
documents[0].page_content[:1000]

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

In [ ]:
chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

In [ ]:
chunks[0].page_content

## Task 4: Embeddings & Vector Store

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
from langchain_community.vectorstores import FAISS

In [ ]:
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("Embeddings stored successfully.")

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [ ]:
query = "What is machine learning?"

results = retriever.invoke(query)
print("Retrieved documents:", len(results))

In [ ]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i + 1} ---")
    print(doc.page_content)

## Task 5: RAG Prompt Template

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a document question-answering assistant.

Answer the user's question ONLY using the provided context.

If the answer is not available in the context, say:
"I don't know."

Do not use outside knowledge.

Context:
{context}
"""
    ),
    ("human", "{question}")
])

In [ ]:
def rag_chat(question):
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )
    messages = rag_prompt.invoke({
        "context": context,
        "question": question
    })
    response = llm.invoke(messages)

    return response.content

In [ ]:
answer = rag_chat("What is machine learning?")

print(answer)

In [ ]:
answer = rag_chat("Explain the main concepts discussed in the document.")

print(answer)

In [ ]:
answer = rag_chat(
    "Who was the first person to walk on Mars?"
)

print(answer)

# PART 3 — ChatGroq RAG Chain
## Task 6: Build RAG Chain

In [ ]:
def rag_chain(question):
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )
    messages = rag_prompt.invoke({
        "context": context,
        "question": question
    })
    response = llm.invoke(messages)

    return response.content

In [ ]:
questions = [
    "What is the main topic of the document?",
    "Explain the most important concept discussed.",
    "What are the key points mentioned in the document?"
]

for question in questions:
    print("\nQuestion:", question)
    print("Answer:", rag_chain(question))

## Task 7: Build Streamlit Chat Interface

In [ ]:
import streamlit as st

from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
import os

st.title("🤖 Groq RAG Chatbot")

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = st.text_input(
        "Enter Groq API Key",
        type="password"
    )

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

if "messages" not in st.session_state:
    st.session_state.messages = []


if "vectorstore" not in st.session_state:
    st.session_state.vectorstore = None

uploaded_file = st.file_uploader("Upload PDF or TXT",type=["pdf", "txt"])

if uploaded_file is not None:
    file_path = uploaded_file.name

    with open(file_path, "wb") as f:
        f.write(uploaded_file.getbuffer())
    if uploaded_file.name.endswith(".pdf"):
        loader = PyPDFLoader(file_path)
    else:
        loader = TextLoader(file_path)

    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    chunks = text_splitter.split_documents(documents)
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    st.session_state.vectorstore = FAISS.from_documents(
        chunks,
        embeddings
    )

    st.success("Document processed successfully!")


for message in st.session_state.messages:

    with st.chat_message(message["role"]):
        st.write(message["content"])


question = st.chat_input(
    "Ask a question about your document..."
)


if question:

    if st.session_state.vectorstore is None:

        st.warning(
            "Please upload a document first."
        )

    else:
        with st.chat_message("user"):
            st.write(question)

        st.session_state.messages.append({"role": "user","content": question})
        retriever = st.session_state.vectorstore.as_retriever(
            search_kwargs={"k": 3}
        )

        retrieved_docs = retriever.invoke(question)
        context = "\n\n".join(
            doc.page_content
            for doc in retrieved_docs
        )

        # RAG prompt
        rag_prompt = ChatPromptTemplate.from_messages([
            (
                "system",
                """You are a document Q&A assistant.

                Answer the question ONLY using the provided context.

                If the answer is not available in the context,
                say "I don't know."

                Context:
                {context}
                """
            ),
            ("human", "{question}")
        ])

        messages = rag_prompt.invoke({
            "context": context,
            "question": question
        })

        response = llm.invoke(messages)

        answer = response.content
        with st.chat_message("assistant"):
            st.write(answer)
        st.session_state.messages.append({
            "role": "assistant",
            "content": answer
        })

## Task 8: Integrate RAG Backend with UI

Run the application : `streamlit run app.py`

# PART 5 — Testing & Validation
## Task 9: Multi-Turn Chat Testing

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [ ]:
import os
import streamlit as st

from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

st.title("🤖 Groq RAG Chatbot")

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = st.text_input(
        "Enter Groq API Key",
        type="password"
    )

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

if "messages" not in st.session_state:
    st.session_state.messages = []


if "vectorstore" not in st.session_state:
    st.session_state.vectorstore = None

uploaded_file = st.file_uploader("Upload PDF or TXT",type=["pdf", "txt"])

if uploaded_file is not None:

    file_path = uploaded_file.name

    with open(file_path, "wb") as f:
        f.write(uploaded_file.getbuffer())
    if uploaded_file.name.endswith(".pdf"):
        loader = PyPDFLoader(file_path)
    else:
        loader = TextLoader(file_path)

    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    chunks = text_splitter.split_documents(documents)
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    st.session_state.vectorstore = FAISS.from_documents(
        chunks,
        embeddings
    )

    st.success("Document processed successfully!")


for message in st.session_state.messages:

    with st.chat_message(message["role"]):
        st.write(message["content"])


question = st.chat_input(
    "Ask a question about your document..."
)


if question:

    if st.session_state.vectorstore is None:

        st.warning("Please upload a document first.")

    else:
        with st.chat_message("user"):
            st.write(question)

        chat_history = []

        for message in st.session_state.messages:
            if message["role"] == "user":
                chat_history.append(HumanMessage(content=message["content"]))
            elif message["role"] == "assistant":
                chat_history.append(AIMessage(content=message["content"]))

        retriever = st.session_state.vectorstore.as_retriever(
            search_kwargs={"k": 3}
        )

        retrieved_docs = retriever.invoke(question)
        context = "\n\n".join(
            doc.page_content
            for doc in retrieved_docs
        )

        rag_prompt = ChatPromptTemplate.from_messages([
            (
                "system",
                """You are a document Q&A assistant.

        Answer the user's question ONLY using the provided context.

        Use the conversation history to understand follow-up questions.

        If the answer is not available in the context, say:
        "I don't know."

        Do not use outside knowledge.

        Context:
        {context}
        """
            ),
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{question}")
        ])

        # Create prompt
        messages = rag_prompt.invoke({
            "context": context,
            "chat_history": chat_history,
            "question": question
        })

        response = llm.invoke(messages)

        answer = response.content
        with st.chat_message("assistant"):
            st.write(answer)
        st.session_state.messages.append({
            "role": "user",
            "content": question
        })

        st.session_state.messages.append({
            "role": "assistant",
            "content": answer
        })

In [ ]:
st.write("Chat history:", st.session_state.messages)

# PART 6 — Mini Project Deliverable
## Task 10: Final ChatGroq RAG App

# Task 11 — Observations & Insights

1. Why is Groq suitable for RAG chatbots?
Groq is useful for RAG applications because it provides fast LLM inference. RAG applications often need to retrieve documents and then generate an answer interactively, so low inference latency can improve the user experience.

2. Difference between Groq RAG and OpenAI RAG
- The RAG architecture is essentially the same:
- The main difference is the LLM provider.
- The retrieval layer can remain the same. 
- The choice between providers depends on requirements such as model availability, latency, pricing, deployment, and data-handling requirements.

3. Role of Streamlit in rapid GenAI prototyping
Streamlit provides a simple way to convert the Python RAG pipeline into an interactive web application.
It provides components such as:
- File upload
- Chat input
- Chat messages
- Session state
- Status/error messages